# HybridGaussianFactorGraph

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridGaussianFactorGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

`HybridGaussianFactorGraph` is a specialized `HybridFactorGraph` designed to represent linearized hybrid systems or systems that are inherently Conditional Linear Gaussian (CLG). It primarily contains:

*   `gtsam.GaussianFactor` (or derived types like `JacobianFactor`, `HessianFactor`)
*   `gtsam.DiscreteFactor` (or derived types like `DecisionTreeFactor`, `TableFactor`)
*   `gtsam.HybridGaussianFactor` (including `HybridGaussianConditional`)

This graph type is typically the result of linearizing a `HybridNonlinearFactorGraph`. It supports specialized elimination methods (`eliminateSequential`, `eliminateMultifrontal` via `EliminateableFactorGraph` base) that correctly handle the mix of discrete and continuous variables, producing a `HybridBayesNet` or `HybridBayesTree`.

In [2]:
import gtsam
import numpy as np

from gtsam import (
    GaussianFactor, JacobianFactor, DecisionTreeFactor,
    HybridGaussianFactorGraph, HybridGaussianFactor, HybridGaussianConditional, GaussianConditional,
    DiscreteValues, VectorValues, HybridValues,
    HybridBayesNet, HybridConditional
)
from gtsam.symbol_shorthand import X, D

## Initialization and Adding Factors

In [3]:
hgfg = HybridGaussianFactorGraph()

# Gaussian Factor (e.g., prior on X0)
prior_model = gtsam.noiseModel.Isotropic.Sigma(1, 1.0)
prior_factor = JacobianFactor(X(0), -np.eye(1), np.zeros(1), prior_model)
hgfg.add(prior_factor)

# Discrete Factor (e.g., prior on D0)
dk0 = (D(0), 2) # D0 has 2 states
discrete_prior = DecisionTreeFactor([dk0], "0.7 0.3") # P(D0=0)=0.7, P(D0=1)=0.3
hgfg.add(discrete_prior)

# Hybrid Gaussian Factor (e.g., measurement on X1 depends on D0)
dk1 = (D(1), 2)
# Measurement model 0: N(X1; 0, 1)
gf0 = JacobianFactor(X(1), np.eye(1), np.zeros(1), gtsam.noiseModel.Isotropic.Sigma(1, 1.0))
# Measurement model 1: N(X1; 2, 0.5)
gf1 = JacobianFactor(X(1), np.eye(1), np.array([2.0]), gtsam.noiseModel.Isotropic.Sigma(1, 0.5))
hybrid_meas = HybridGaussianFactor(dk1, [gf0, gf1])
hgfg.add(hybrid_meas)

# Hybrid Gaussian Conditional (Result of previous elimination, P(X0 | D0, X1))
# Construct simplified P(X0|D0) for demo
# Mode 0: X0 = N(0, 1) -> R=1, d=0
gc0 = GaussianConditional(X(0), np.zeros(1), np.eye(1), gtsam.noiseModel.Unit.Create(1))
# Mode 1: X0 = N(1, 2) -> R=1/sqrt(2), d=1/sqrt(2)
gc1 = GaussianConditional(X(0), np.array([1.0]), np.eye(1)/np.sqrt(2.0), gtsam.noiseModel.Isotropic.Sigma(1, np.sqrt(2.0)))

# Create HybridGaussianConditional P(X0 | D0)
# NOTE: This example is simplified; a real conditional might have more parents.
# This creates P(X0|D(0))
# hgc = HybridGaussianConditional(dk0, [gc0, gc1]) # TODO: Check constructor signature / simple way
# hgfg.add(hgc) # Conditionals can also be added

hgfg.print("\nHybrid Gaussian Factor Graph:")

AttributeError: 'gtsam.gtsam.HybridGaussianFactorGraph' object has no attribute 'add'

## Key Inspection and Choosing a Gaussian Subgraph

Besides the standard `keys()`, `discreteKeys()`, etc., you can `choose` a specific `GaussianFactorGraph` corresponding to a discrete assignment.

In [4]:
print(f"\nDiscrete Keys: {hgfg.discreteKeys()}")
print(f"Continuous Keys: {hgfg.continuousKeySet()}")

# Choose the GaussianFactorGraph for a specific assignment
assignment = gtsam.DiscreteValues()
assignment[D(0)] = 0
assignment[D(1)] = 1 # Selects gf1 from hybrid_meas

gaussian_subgraph = hgfg.choose(assignment)

print(f"\nChosen GaussianFactorGraph for D0=0, D1=1:")
gaussian_subgraph.print()
# Note: The discrete factor is ignored in the chosen graph.
# The HybridGaussianFactor contributes its gf1 component.
# The prior on X0 is included.

AttributeError: 'gtsam.gtsam.HybridGaussianFactorGraph' object has no attribute 'discreteKeys'

## Computing Errors (`error`, `errorTree`, `probPrime`, `discretePosterior`)

Several methods exist to evaluate the graph under different assumptions.

In [5]:
# --- Error methods ---
hybrid_values = HybridValues()
hybrid_values.insert(X(0), np.array([0.1]))
hybrid_values.insert(X(1), np.array([2.1])) # Close to mean of gf1
hybrid_values.insert(assignment) # Use D0=0, D1=1

# Sum of errors (Gaussian parts) + neg log prob (Discrete parts)
total_error = hgfg.error(hybrid_values)
print(f"\nTotal Hybrid Error (Gaussian+Discrete): {total_error}") # Should be low

# Error Tree (only considers Gaussian factors, returns ADT over discrete keys)
continuous_values = hybrid_values.continuous()
error_tree = hgfg.errorTree(continuous_values)
print("\nError Tree (Gaussian factor errors):")
error_tree.print()

# probPrime (unnormalized posterior density for a full hybrid assignment)
prob_prime = hgfg.probPrime(hybrid_values)
print(f"\nProbability Prime (unnormalized density): {prob_prime}") # exp(-error)

# discretePosterior (P(M | X=x), needs only continuous values)
discrete_posterior_tree = hgfg.discretePosterior(continuous_values)
print("\nDiscrete Posterior Tree P(M | X=x):")
discrete_posterior_tree.print() # ADT representing normalized discrete posterior

NameError: name 'assignment' is not defined

## Elimination (`eliminateSequential`, `eliminateMultifrontal`)

Elimination methods handle the hybrid nature, producing `HybridBayesNet` or `HybridBayesTree`. Discrete keys are typically eliminated *after* continuous keys by default using `HybridOrdering`.

In [6]:
# Default ordering eliminates continuous then discrete
ordering = gtsam.HybridOrdering(hgfg)
print(f"\nDefault Hybrid Ordering: {ordering}") # Expect X(0), X(1), D(0), D(1)...

try:
    # Sequential elimination
    hybrid_bayes_net: gtsam.HybridBayesNet
    hybrid_bayes_net, remaining_factors = hgfg.eliminatePartialSequential(ordering)
    print("\nResulting HybridBayesNet:")
    hybrid_bayes_net.print()
    print("\nRemaining Factors:")
    remaining_factors.print()

    # Multifrontal elimination (often preferred)
    hybrid_bayes_tree = hgfg.eliminateMultifrontal(ordering)
    print("\nResulting HybridBayesTree:")
    hybrid_bayes_tree.print()

except Exception as e:
    print(f"\nError during elimination: {e}")
    print("(Elimination might require specific graph structures or handling)")

AttributeError: module 'gtsam' has no attribute 'HybridOrdering'